# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

## Business Scenario

You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.


Your task is to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Dataset Description: Hotel Bookings

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance

### Data Dictionary

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Load the `hotels.csv` file from https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/hotels.csv
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [4]:
import pandas as pd

# Load the dataset
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/hotels.csv"
df = pd.read_csv(url)

# 1. How many total rows and columns are in the dataset?
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

# 2. What types of features (categorical, numerical) are included?
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print("\nNumerical features:")
for feature in numerical_features:
    print(f"- {feature}")

print("\nCategorical features:")
for feature in categorical_features:
    print(f"- {feature}")

# Display the first few rows of the dataframe to get a glance
# print(df.head())

# Store the dataframe for later use in the notebook
get_ipython().user_ns['df'] = df

Total rows: 119390
Total columns: 32

Numerical features:
- is_canceled
- lead_time
- arrival_date_year
- arrival_date_week_number
- arrival_date_day_of_month
- stays_in_weekend_nights
- stays_in_week_nights
- adults
- children
- babies
- is_repeated_guest
- previous_cancellations
- previous_bookings_not_canceled
- booking_changes
- agent
- company
- days_in_waiting_list
- adr
- required_car_parking_spaces
- total_of_special_requests

Categorical features:
- hotel
- arrival_date_month
- meal
- country
- market_segment
- distribution_channel
- reserved_room_type
- assigned_room_type
- deposit_type
- customer_type
- reservation_status
- reservation_status_date


In [5]:
# Impute missing values
# For 'children', fill NaN with 0 (as it represents count)
df['children'].fillna(0, inplace=True)

# For 'country', fill NaN with 'Unknown' as it's a categorical feature
df['country'].fillna('Unknown', inplace=True)

# For 'agent' and 'company', fill NaN with 0 and convert to int (assuming 0 means no agent/company)
df['agent'].fillna(0, inplace=True)
df['company'].fillna(0, inplace=True)
df['agent'] = df['agent'].astype(int)
df['company'] = df['company'].astype(int)

print(f"DataFrame shape after imputation: {df.shape}")
print("Missing values after imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

DataFrame shape after imputation: (119390, 32)
Missing values after imputation:
Series([], dtype: int64)


/tmp/ipython-input-428029543.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['children'].fillna(0, inplace=True)
/tmp/ipython-input-428029543.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.meth

In [6]:
# Identify categorical features again (after imputation, just to be safe)
categorical_cols = df.select_dtypes(include=['object']).columns

# Apply one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"DataFrame shape after one-hot encoding: {df.shape}")
print("First 5 rows after encoding:")
print(df.head())

DataFrame shape after one-hot encoding: (119390, 1176)
First 5 rows after encoding:
   is_canceled  lead_time  arrival_date_year  arrival_date_week_number  \
0            0        342               2015                        27   
1            0        737               2015                        27   
2            0          7               2015                        27   
3            0         13               2015                        27   
4            0         14               2015                        27   

   arrival_date_day_of_month  stays_in_weekend_nights  stays_in_week_nights  \
0                          1                        0                     0   
1                          1                        0                     0   
2                          1                        0                     1   
3                          1                        0                     1   
4                          1                        0                     2 

In [7]:
# Create X (features) and y (target)
y = df['is_canceled']

# Identify columns related to reservation status that would cause data leakage
leakage_cols = [col for col in df.columns if 'reservation_status' in col and col != 'is_canceled']

# Drop 'is_canceled' and all identified leakage columns from X
X = df.drop(columns=['is_canceled'] + leakage_cols)

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")
print("First 5 rows of X:")
print(X.head())
print("First 5 values of y:")
print(y.head())

Shape of X: (119390, 248)
Shape of y: (119390,)
First 5 rows of X:
   lead_time  arrival_date_year  arrival_date_week_number  \
0        342               2015                        27   
1        737               2015                        27   
2          7               2015                        27   
3         13               2015                        27   
4         14               2015                        27   

   arrival_date_day_of_month  stays_in_weekend_nights  stays_in_week_nights  \
0                          1                        0                     0   
1                          1                        0                     0   
2                          1                        0                     1   
3                          1                        0                     1   
4                          1                        0                     2   

   adults  children  babies  is_repeated_guest  ...  assigned_room_type_H  \
0       2     

In [8]:
from sklearn.model_selection import train_test_split

# Split the data into training and test sets (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (83573, 248)
Shape of X_test: (35817, 248)
Shape of y_train: (83573,)
Shape of y_test: (35817,)


In total, there were 119,390 rows and 32 columns in this dataset.


There were both numerical and categorical features, with more numerical features than categorical.


To clean and prepare this dataset, I imputed missing values. I also one-hot encoded categorical features so that they became numerical. This will allow these features to be modeled. Last, I split the data into training and testing sets.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [9]:
from sklearn.naive_bayes import GaussianNB

# Initialize the Naïve Bayes classifier
naive_bayes_model = GaussianNB()

# Train the model on the training data
naive_bayes_model.fit(X_train, y_train)

print("Naïve Bayes model trained successfully.")

Naïve Bayes model trained successfully.


In [10]:
from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on the test data
y_pred_nb = naive_bayes_model.predict(X_test)

print("Naïve Bayes Model Performance:\n")

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_nb))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

Naïve Bayes Model Performance:

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.33      0.47     22478
           1       0.45      0.91      0.60     13339

    accuracy                           0.54     35817
   macro avg       0.65      0.62      0.54     35817
weighted avg       0.71      0.54      0.52     35817


Confusion Matrix:
[[ 7321 15157]
 [ 1154 12185]]


The model performs decently well, but not super well. The maco average F1 score is 54%, which tells me that model is doing okay overall, but likely struggles with minority classes. The F1 score is best used to judge performance because it gives a balanced measure of classification quality.


This model could help the hotel predict cancellations in their system before they happen. This model is better at predicting reservations that are canceled as opposed to reservations that are not canceled. 60% vs 47%, accordding to the F1 scores. If this model can correctly predict cancellations 60% of the time, it could help the company plan for cancellations.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [11]:
from sklearn.svm import LinearSVC

# Initialize the Linear SVM classifier
# Note: LinearSVC is specifically for linear SVM and is optimized for large datasets.
# For non-linear kernels, one would use SVC with 'kernel="linear"' or other kernels.
svm_model = LinearSVC(random_state=42, dual=False) # dual=False is recommended for n_samples > n_features

# Train the model on the training data
svm_model.fit(X_train, y_train)

print("Linear SVM model trained successfully.")

Linear SVM model trained successfully.


In [12]:
from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on the test data using the trained SVM model
y_pred_svm = svm_model.predict(X_test)

print("Linear SVM Model Performance:\n")

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_svm))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

Linear SVM Model Performance:

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.92      0.86     22478
           1       0.83      0.63      0.72     13339

    accuracy                           0.81     35817
   macro avg       0.82      0.78      0.79     35817
weighted avg       0.82      0.81      0.81     35817


Confusion Matrix:
[[20728  1750]
 [ 4904  8435]]


This model performs very well. The macro average F1 score is 79%, which is pretty good. The weighted average F1 score is 81%, which is even better. The F1 scores for both prediction categories are in the 70's and 80's, which is good. I like using the F1 score to evaluate models because F1 only becomes high when both precision and recall are good, so it is a helpful way to get an idea of the overall effectiveness of a model.


SVM could provide better insight than simpler models when business data is not totally independent. It can also be better when the boundary between classes is complex, or non-linear. It can also be good for business situations where classes overlap heavily within the data. SVM is more intensive than simpler models, but can provide better predictions.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLBClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Evaluate accuracy and performance

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [13]:
from sklearn.neural_network import MLPClassifier

# Build a MLPClassifier model with 2 hidden layers
# Example architecture: (100, 50) means 2 hidden layers, first with 100 neurons, second with 50
# You can adjust these values based on performance and complexity requirements.
mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)

# Train the model
mlp_model.fit(X_train, y_train)

print("Neural Network (MLPClassifier) model trained successfully.")

Neural Network (MLPClassifier) model trained successfully.


In [14]:
from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on the test data using the trained MLPClassifier model
y_pred_nn = mlp_model.predict(X_test)

print("Neural Network (MLPClassifier) Model Performance:\n")

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_nn))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nn))

Neural Network (MLPClassifier) Model Performance:

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.90      0.89     22478
           1       0.83      0.79      0.81     13339

    accuracy                           0.86     35817
   macro avg       0.85      0.85      0.85     35817
weighted avg       0.86      0.86      0.86     35817


Confusion Matrix:
[[20308  2170]
 [ 2843 10496]]


This model is more accurate than the other two models, across the board. Its accurace score is 86%, and its macro average F1 score is 85%. These are very good metrics.


I could see the company being comfortable using a black box model like this, if the accuracy was good enough, which it seems like it is. Management just needs to trust the model. However, this would not be a suitable model if the company's management wanted a simple, auditable model that is easy to understand.




## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [15]:
from sklearn.metrics import accuracy_score

# Calculate accuracy for Naïve Bayes
accuracy_nb = accuracy_score(y_test, y_pred_nb)

# Calculate accuracy for SVM
accuracy_svm = accuracy_score(y_test, y_pred_svm)

# Calculate accuracy for Neural Network
accuracy_nn = accuracy_score(y_test, y_pred_nn)

print(f"Naïve Bayes Model Accuracy: {accuracy_nb:.4f}")
print(f"SVM Model Accuracy: {accuracy_svm:.4f}")
print(f"Neural Network Model Accuracy: {accuracy_nn:.4f}")

print("\n--- Model Comparison ---")

if accuracy_nn >= accuracy_svm and accuracy_nn >= accuracy_nb:
    print("The Neural Network model performed best overall in terms of accuracy.")
elif accuracy_svm >= accuracy_nn and accuracy_svm >= accuracy_nb:
    print("The SVM model performed best overall in terms of accuracy.")
else:
    print("The Naïve Bayes model performed best overall in terms of accuracy.")

Naïve Bayes Model Accuracy: 0.5446
SVM Model Accuracy: 0.8142
Neural Network Model Accuracy: 0.8600

--- Model Comparison ---
The Neural Network model performed best overall in terms of accuracy.


The neural network model was slow compared to the others, but its accuracy was much higher than the others. The neural network model wins in terms of accuracy alone. The others were easier to interpret and faster, but you can't beat 86% accuracy with these three models. The neural network model wins.


I would recommend this model for deployment just because it is so accurate. It does take a while to complete, but 86% accuracy is very hard to argue with. I would push to deploy this model.

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


I recommend implementing the neural network model to predict which bookings will be canceled. This model helps predict which bookings will be canceled or not canceled, which will help your company streamline operations and have more accurate forecasting capabilities. You can adjust availability of rooms on your site and how many rooms you actually put up for sale based on this model's output. The one limitation of the neural network model is that it takes a long time to run. However, its high accuracy score is worth this trade-off. It is a very complex model whose inner-workings are hard to understand, but it has an extremely high accuracy score. More data will always improve the accuracy of the model. I would focus on finding data pertaining to "special cases" such as people who don't normally stay at your hotels, or huge groups/solo travelers. Diverse data will make the model even more accurate.



This assignment directly relates to the customizd learning outcome I created in Canvas. I mentioned wanting to analyze hotel booking data and make recommendations, and this assignment allowed me to do that.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [16]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 343718 bytes to assignment_12_bayes_svm_neural.html


# Task
Load the `hotels.csv` dataset from "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/hotels.csv", print its total number of rows and columns, and then identify and list the numerical and categorical features.

## Load Data, Get Shape, and Identify Feature Types

### Subtask:
Load the `hotels.csv` dataset, print its total number of rows and columns, and then identify and list the numerical and categorical features.


## Summary:

### Q&A
The `hotels.csv` dataset contains 119,390 rows and 32 columns.

The numerical features are: `lead_time`, `arrival_date_year`, `arrival_date_week_number`, `arrival_date_day_of_month`, `stays_in_weekend_nights`, `stays_in_week_nights`, `adults`, `children`, `babies`, `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `booking_changes`, `agent`, `company`, `days_in_waiting_list`, `adr`, `required_car_parking_spaces`, `total_of_special_requests`.

The categorical features are: `hotel`, `arrival_date_month`, `meal`, `country`, `market_segment`, `distribution_channel`, `reserved_room_type`, `assigned_room_type`, `deposit_type`, `customer_type`, `reservation_status`, `reservation_status_date`.

### Data Analysis Key Findings
*   The `hotels.csv` dataset consists of 119,390 entries and 32 distinct features.
*   The dataset is composed of 19 numerical features and 12 categorical features.

### Insights or Next Steps
*   The identified feature types provide a clear roadmap for subsequent data preprocessing steps, such as handling missing values, encoding categorical variables, and scaling numerical features.
*   Given the mix of numerical and categorical data, initial descriptive statistics and visualizations for each feature type would be beneficial to understand data distributions and potential outliers.
